# Polyhedral decomposition widget

Build a small ReLU network, then drag the weight/bias sliders and watch the polyhedral
decomposition of the input plane (and its dual graph) update live.

The Python implementation lives in `src/` and mirrors the JS modules in `js/`.

In [1]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

import sys, pathlib
repo_root = pathlib.Path.cwd()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src import PolyhedraWidget, pp

In [2]:
# Architecture: input layer must be size 2 so the decomposition lives in the plane.
# Try e.g. [2, 3, 1], [2, 4, 1], or [2, 4, 3, 1].
widget = PolyhedraWidget(architecture=[2, 4, 3, 1], seed=0)
widget

**How the decomposition is computed**

Each ReLU hidden neuron contributes a single affine constraint on its region: within a region where every *earlier* neuron has a known ON/OFF state, the pre-activation is an affine function of the input, so its activation boundary is a straight line. We start from the viewing rectangle and split every current region by that line, tracking the running affine map from input to post-ReLU activation. The result is the exact polyhedral decomposition — no grid sampling, no resolution knob.

- *Randomize* / *Zero* mutate the network and resync the sliders. *View ±* adjusts the bounding rectangle; black lines are neuron activation boundaries clipped to their region.
- Pass show_dual_graph=True to the constructor to render the dual graph next to the decomposition.

## Exporting weights & biases

Every widget exposes its parameters as a plain Python dict via `widget.state`. Use `pp(state, json=True)` to print it in the canonical JSON save format — matching `multilayer_network.json` byte-for-byte: scalar-only lists stay on one line, lists-of-lists expand with two-space indent. The same output lands in the Textarea next to the sliders when you click **Export weights & biases**, ready to copy-paste into a `.json` file. To read that dict back, feed it to `PolyhedraWidget(state=...)` or `MultiLayerNetwork.from_state(...)`.

In [3]:
state = widget.state
pp(state, json=True)

{
  "architecture": [2, 4, 3, 1],
  "weights": [
    [
      [0.2739233746429086, -0.4604265724722594],
      [-0.9180529521276106, -0.9669447289429418],
      [0.6265404784005448, 0.8255111545554434],
      [0.21327155153435973, 0.4589931219679968]
    ],
    [
      [0.7148085531751387, -0.9328288493890713, 0.45931089285988813, -0.648688758794882],
      [0.7263578446997732, 0.08292244049818343, -0.40057621892523043, -0.1546255576046831],
      [-0.9433606577090741, -0.7514334470008721, 0.34124882938726064, 0.2943790231485002]
    ],
    [
      [0.9616706775524602, 0.3710839689613894, 0.3009185525356326]
    ]
  ],
  "biases": [
    [
      [0.08724998293084574],
      [0.8701448475755365],
      [0.6317071082430643],
      [-0.9945229996597038]
    ],
    [
      [0.2307702229625077],
      [-0.23264489147623313],
      [0.994419871578422]
    ],
    [
      [0.37689346114188016]
    ]
  ],
  "data": []
}


### Modify and re-plot

You can edit any entry in the exported dict and spin up a new widget from it — the sliders will start at the modified values.

In [4]:
# Take the exported state, double the first-layer weights, and zero out the output bias.
import copy
modified = copy.deepcopy(state)
modified["weights"][0] = [[2 * v for v in row] for row in modified["weights"][0]]
modified["biases"][-1] = [0.0 for _ in modified["biases"][-1]]

widget_modified = PolyhedraWidget(state=modified)
widget_modified